# Week 7
## E-commerce Orders Delta Pipeline

In [0]:
import yaml
local_config_path = "/Workspace/Repos/adb-tetiana@startsteps.org/E-commerce-Orders-Delta-Pipeline/config/projects_config.yml"

with open(local_config_path, "r") as f:
    config = yaml.safe_load(f)

catalog_name = config["catalog"]
schema_name = config["schema"]   
landing_volume = config["landing_volume"]
checkpoint_volume = config["checkpoint_volume"]

landing_path = config["paths"]["landing_orders"]
checkpoint_path = config["paths"]["autoloader_checkpoint"]

bronze_table = config["tables"]["bronze_orders"]


print(f"landing_path: {landing_path}")
print(f"checkpoint_path: {checkpoint_path}")
print(f"bronze_table: {bronze_table}")


## Step - 2 Land the first raw file in the volume

In [0]:
dbutils.fs.mkdirs(landing_path)

repo_sample_path = "file:/Workspace/Repos/adb-tetiana@startsteps.org/E-commerce-Orders-Delta-Pipeline/sample_data/"

dbutils.fs.cp(
    f"{repo_sample_path}/batch_1_orders.csv",
    f"{landing_path}/batch_1_orders.csv"
)    

display(dbutils.fs.ls(landing_path))


In [0]:
raw_preview_df = (spark.read
                    .option("header", True)
                    .option("inferSchema", True)
                    .csv(f"{landing_path}/batch_1_orders.csv")
)                    

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType,
    TimestampType
)

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("order_date", DateType(), True),
    StructField("shipping_method", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("delivery_country", StringType(), True),
    StructField("last_updated_ts", TimestampType(), True)
])



## Step -3 Use Auto Loader to ingest incrementaly into Delta

In [0]:
# defined the injections source
orders_stream = (
    spark.readStream
    .format("cloudFiles") # format files for autoloader
    .schema(schema)
    .option("cloudFiles.format","csv") # type of the loaded files
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema") # where to take source data 
    .option("header", "true") # tells autoloader that the first row in file  contains colum names
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
)

# defines the output behaviour
query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint_path + "/schema") # where to write to, remembers the injestion process 
    .trigger(availableNow=True) # define, often process files; availableNow - tell to process what currently avialiable in the source
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
%sql
USE retail_dev.orders;

SELECT * FROM bronze_orders LIMIT 10

## Step 4 - Why Delta matters

In [0]:
# test 
from pyspark.sql import  Row

bad_schema_df = spark.createDataFrame(
    [
        Row(
            order_id = "09999", 
            customer_id = "C900", 
            order_status = "SHIPPED",
            order_amount = "NOT A NUMBER",
            order_date = "2026-01-01",
            shipping_method = "STANDARD",
            payment_status = "PAID",
            delivery_country = "DE",
            last_updated_ts = "2026-01-01T00:00:00.000+0000"
            )
    ]
)

bad_schema_df.write.mode("append").saveAsTable(bronze_table)

## MERGE Section

In [0]:
dbutils.fs.cp(
    f"{repo_sample_path}/order_corrections.csv",
    f"{landing_path}/order_corrections.csv"
)    

correction_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(f"{landing_path}/order_corrections.csv")

)

correction_df.createOrReplaceTempView("order_corrections")
display(correction_df)

In [0]:
%sql
USE retail_dev.orders;

MERGE INTO bronze_orders AS target
USING order_corrections AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET 
  target.order_status = source.order_status,
  target.order_amount = source.order_amount,
  target.payment_status = source.payment_status,
  target.last_updated_ts = source.last_updated_ts     
WHEN NOT MATCHED THEN INSERT *;

## Below are the helpers

In [0]:
%sql
USE retail_dev.orders;

DROP TABLE IF EXISTS bronze_orders;

In [0]:
dbutils.fs.rm(checkpoint_path, True)